# Multilayer Perceptron (MLP) with PyTorch on MNIST
## Handwritten Digit Classification

After implementing a Multilayer Perceptron (MLP) from scratch, it is highly beneficial to reproduce the same model using **PyTorch**. This serves two main purposes:

1. **Validation**: checking correctness against a widely used deep learning framework.
2. **Extensibility**: establishing a baseline for future experiments such as regularization, improved optimizers, better architectures, and GPU acceleration.

This notebook provides a conceptual introduction to how a neural network is designed and trained in `torch.nn`, using an MLP for MNIST as the running example.

## The Core Idea in PyTorch: `nn.Module`

The central abstraction in PyTorch neural networks is **`nn.Module`**.

- A **layer** such as `nn.Linear`, `nn.ReLU`, or `nn.Conv2d` is a subclass of `nn.Module`.
- A **model** is also a subclass of `nn.Module`.
- A model is usually built by **combining smaller Modules** into a larger Module.

So in PyTorch, a neural network is best viewed as a **tree of Modules**.

`nn.Module` is not just a data container. It is a base class that provides the standard behavior needed by neural networks, including:

- registration of submodules
- registration of learnable parameters
- device transfer (`.to(device)`, `.cuda()`, `.cpu()`)
- training/evaluation mode switching (`.train()`, `.eval()`)
- compatibility with optimizers and autograd
- saving and loading through `state_dict()`

A useful analogy is **LEGO**:

- `nn.Module` is like the **standard design system** that all LEGO parts must follow.
- Layers such as `Linear` and `ReLU` are like **individual LEGO pieces**.
- A model is like the **final structure assembled from those pieces**.

This is why custom models usually inherit from `nn.Module`: doing so makes the model understandable to the rest of PyTorch.

##  Parameters, Tensors, and Outputs

A key conceptual distinction in PyTorch is the difference between **parameters** and **outputs**.

### Tensor
A `Tensor` is the basic numerical object in PyTorch. It may store data, intermediate results, or gradients.

### Parameter
A `Parameter` is a special kind of tensor used to represent **learnable state** inside a model. Examples include the `weight` and `bias` of a linear layer.

A parameter matters because it is **registered** inside a Module. Once registered:

- it appears in `model.parameters()`
- the optimizer can update it
- it is saved in `state_dict()`
- it moves with the model to CPU/GPU

### Output
The `output` of a model is **not** a parameter. It is the result of applying the model to the current input.

In other words:

```text
parameters = internal learnable state
output     = computed result for a particular input
```

Training updates the **parameters**, not the output, and not the model structure itself.

##  A General Workflow for Designing and Training a Model in `torch.nn`

A typical PyTorch pipeline looks like this:

1. import the required packages
2. Define the device (`cuda` if available, otherwise `cpu`)
3. Prepare the dataset using `torch.utils.data.Dataset`
4. Split the dataset into training and testing sets
5. Create `DataLoader` objects for batching and iteration
6. Define the model by inheriting from `nn.Module`
7. Define the loss function and optimizer
8. Write a training loop for one epoch
9. Repeat training for several epochs
10. Write a test/evaluation function

The remainder of this notebook explains the conceptual roles of each part.


## 1. Imports 

In [6]:
# pip install torch torchvision
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms


## 2. Defining the Device

A standard pattern in PyTorch is:


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"


This allows the same code to run on GPU if available, otherwise on CPU.
The model and data should be moved to the same device before computation.

## 3,4. Load the MNIST Dataset with `torchvision`

MNIST images are 28×28 grayscale. For an MLP, we flatten each image into a 784-dimensional vector.
We will use `datasets` from `torchvision` to load the [MNIST](https://yann.lecun.com/exdb/mnist/) handwritten digits dataset. You can find the list of datasets available on torchvision [here](https://pytorch.org/vision/0.8/datasets.html). Now let's take a loot at the parameters we set:


*   `root` sets the directory we store and load our data from.
*   `train` indicates wether we want the training dataset or the test dataset.
*   `transform` allows us to apply transformations to our data, here we are only going to convert the data to tensor so that they work with PyToch, however in the future notebooks you will see more complicated transformations.

We do not need to split the data into training and testing sets here, as this has already been handled by `torchvision`. Additionally, the data is already in the `Dataset` class format; otherwise, we would need to implement a function to convert our data into a `Dataset` object.

In [8]:
transform = transforms.Compose([
    transforms.ToTensor()
])

train_dataset = datasets.MNIST(
    root='data', train=True, download=True, transform=transform
)

test_dataset = datasets.MNIST(
    root='data', train=False, download=True, transform=transform
)

print(f"Training data: {train_dataset}\n")
print(f"Test data: {test_dataset}")

Training data: Dataset MNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: Compose(
               ToTensor()
           )

Test data: Dataset MNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: Compose(
               ToTensor()
           )


## 5. `DataLoader` object for batching and iteration

PyTorch uses two important abstractions for data handling:

- **`Dataset`**: defines how to access individual samples
- **`DataLoader`**: creates an iterable over the dataset, handling batching, shuffling, and efficient loading

Conceptually:

```text
raw data → Dataset → DataLoader → batches of (data, target)
```

This makes it easy to iterate through the training set batch by batch.




To make loading and working with the data easier, we are going to use `DataLoader` from `torch.utils.data`. The `DataLoader` takes in a dataset and a `batch_size` parameter, and allows us to iterate over the dataset. Here we do one iteration just to see the data shapes:

In [9]:
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

## 6. Define the MLP model by inheriting from `nn.Module`

This is a standard fully connected network: 784 → hidden → hidden → 10.
We do not apply softmax inside the model because CrossEntropyLoss expects raw logits.


In [14]:
class SimpleMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 20),
            nn.ReLU(),
            nn.Linear(20, 10)
        )

    def forward(self, x):
        return self.layers(x)

model = SimpleMLP().to(device)
model


SimpleMLP(
  (layers): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=784, out_features=20, bias=True)
    (2): ReLU()
    (3): Linear(in_features=20, out_features=10, bias=True)
  )
)

Two methods are central:

- **`__init__`**: defines the structure of the model
- **`forward`**: defines how data flows through the model

Notice that `nn.Sequential` is itself a **Module container**. It stores several Modules and applies them one after another.

Important distinction:

- `Linear` has parameters (`weight`, `bias`)
- `ReLU` and `Flatten` do not have parameters
- all of them are still Modules

### Module Hierarchy Diagram

For the `SimpleMLP` model, the Module tree looks like this:

```text
SimpleMLP (nn.Module)
│
└── layers (nn.Sequential)                         ← container Module
    │
    ├── (0) Flatten                                ← Module, no parameters
    │
    ├── (1) Linear(28*28 → 20)                     ← Module
    │       ├── weight : Parameter [20, 784]
    │       └── bias   : Parameter [20]
    │
    ├── (2) ReLU                                   ← Module, no parameters
    │
    └── (3) Linear(20 → 10)                        ← Module
            ├── weight : Parameter [10, 20]
            └── bias   : Parameter [10]
```

This makes it clear that the **model is not separate from the layers**. The model is the outer Module that contains the inner Modules.

### Conceptual Hierarchy in PyTorch

The hierarchy of concepts is:

```text
Tensor
  ↓
Parameter                          ← learnable tensor
  ↓
Module (nn.Module)                 ← standard neural-network building block
  ↓
Layer (Linear, ReLU, Conv2d, ...)
  ↓
Model (composition of layers / modules)
```

So a model is best understood as a higher-level Module built from lower-level Modules.


### What Is Stored Inside a Module?

A Module conceptually manages three kinds of internal state:

```text
Model
 ├── Submodules (_modules)
 │     ├── layers
 │     ├── fc1
 │     └── relu
 │
 ├── Parameters (_parameters)
 │     ├── weight
 │     └── bias
 │
 └── Buffers (_buffers)
       └── non-trainable state (e.g. BatchNorm running statistics)
```

This is why a registered `Parameter` is different from an ordinary tensor. A plain tensor is just an attribute unless PyTorch is told to treat it as part of the model.


## 7. Define the loss function and optimizer

Once the model is defined, we instantiate a loss function and an optimizer.

### Loss Function

```python
criterion = nn.CrossEntropyLoss()
```

`CrossEntropyLoss` is a class that computes the difference between the model's predicted logits and the correct class labels.

Conceptually:

```text
output + target → loss
```

Its role is to measure **how wrong** the model currently is.

### Optimizer

```python
optimizer = torch.optim.Adam(model.parameters())
```

The optimizer is responsible for updating the model's parameters during training.

A very important point is that the optimizer receives:

```text
model.parameters()
```

This means it holds **references** to the actual `Parameter` objects stored inside the model. It does not make an independent copy of them.

Therefore, when the optimizer updates a parameter, it is directly updating the parameter inside the model.


In [15]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

### Forward Flow Diagram

For one batch of images, the forward computation looks like this:

```text
input image x
   shape: [batch_size, 1, 28, 28]
            │
            ▼
Flatten
   shape: [batch_size, 784]
            │
            ▼
Linear(784 → 20)
   shape: [batch_size, 20]
            │
            ▼
ReLU
   shape: [batch_size, 20]
            │
            ▼
Linear(20 → 10)
   shape: [batch_size, 10]
            │
            ▼
output logits
```

The `output` here is the prediction for the current input batch. It is **not** the stored state of the model.


## 8. Write a training loop for one epoch

A typical one-epoch training function is conceptually organized as follows:

```python
def train(data_loader, model, criterion, optimizer):
    model.train()

    for data, target in data_loader:
        data = data.to(device)
        target = target.to(device)

        output = model(data)
        loss = criterion(output, target)

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
```

The meaning of each step is:

- `model.train()` puts the model into training mode
- `output = model(data)` performs the forward pass
- `loss = criterion(output, target)` computes the error
- `loss.backward()` computes gradients for each parameter
- `optimizer.step()` updates the model parameters
- `optimizer.zero_grad()` clears old gradients before the next iteration

The crucial point is:

```text
the actual parameter update happens at optimizer.step()
```

By contrast:

```text
loss.backward() computes gradients, but does not change weights
```

### Training Flow Diagram

The full training flow for one batch is:

```text
batch of data, target
        │
        ▼
 data = data.to(device)
 target = target.to(device)
        │
        ▼
output = model(data)                    ← forward pass
        │
        ▼
loss = criterion(output, target)        ← compute loss
        │
        ▼
loss.backward()                         ← compute gradients into param.grad
        │
        ▼
optimizer.step()                        ← update model parameters
        │
        ▼
optimizer.zero_grad()                   ← clear gradients
        │
        ▼
next batch
```

### Parameter Update Mechanism

The role of the optimizer can be summarized as:

```text
model.parameters()
      │
      ▼
optimizer holds references to these Parameter objects
      │
      ▼
loss.backward() fills:
    parameter.grad
      │
      ▼
optimizer.step() updates:
    parameter.data
```

So the overall mechanism is:

```text
gradient is computed in backward()
weights are changed in step()
```

This explains why the model object itself remains the same object throughout training, while its **internal state** changes.

### Difference Between Model, Parameters, and Output

The following summary is especially useful:

```text
model       = the object / structure
parameters  = internal learnable state of the model
output      = result of applying the model to input
```

or equivalently:

```text
input  ──>  model(using parameters)  ──>  output
                         ▲
                         │
                  optimizer updates
```

This distinction is essential when reading training code.


In [18]:
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        correct += (logits.argmax(1) == y).sum().item()
        total += y.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)     # Copy data and targets to GPU
            logits = model(x)                     # Do a forward pass
            loss = criterion(logits, y)           # Calculate the loss
            total_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += y.size(0)
    return total_loss / total, correct / total


## 9. Repeat training and evaluation on validation for several epochs

In [19]:
epochs = 5
for epoch in range(1, epochs + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    test_loss, test_acc = evaluate(model, test_loader, criterion, device)
    print(f'Epoch {epoch:02d} | Train Acc: {train_acc:.4f} | Validation Acc: {test_acc:.4f}')

Epoch 01 | Train Acc: 0.8462 | Test Acc: 0.9165
Epoch 02 | Train Acc: 0.9208 | Test Acc: 0.9306
Epoch 03 | Train Acc: 0.9323 | Test Acc: 0.9344
Epoch 04 | Train Acc: 0.9383 | Test Acc: 0.9413
Epoch 05 | Train Acc: 0.9430 | Test Acc: 0.9440


##  One Compact End-to-End Diagram

The entire process can be summarized in one diagram:

```text
                 ┌─────────────────────────────┐
                 │       SimpleMLP Model       │
                 │        (nn.Module)          │
                 └─────────────┬───────────────┘
                               │
                               ▼
                      [Flatten → Linear → ReLU → Linear]
                               │
                               ▼
                           output logits
                               │
                               ▼
                     CrossEntropyLoss(output, target)
                               │
                               ▼
                          scalar loss
                               │
                               ▼
                         loss.backward()
                               │
                               ▼
                    gradients stored in param.grad
                               │
                               ▼
                         optimizer.step()
                               │
                               ▼
                    parameters inside model updated
```


##  Final Takeaways

- `nn.Module` is the foundational abstraction in PyTorch neural network design.
- Layers are Modules, and a model is a larger Module composed of smaller Modules.
- Parameters are the learnable internal state stored inside layers.
- Outputs are computed results, not stored learnable state.
- `loss.backward()` computes gradients.
- `optimizer.step()` performs the actual parameter update.
- Training changes the **internal state** of the model, not the model's identity or structure.

A final sentence worth remembering is:

> **Training in PyTorch means updating the internal parameters of a model, not changing the model itself.**
